# Assignment 2: Mini Project 1 - Data Science with Generative AI
This notebook builds a complete end-to-end data science pipeline using the **Student Performance** dataset from the UCI Machine Learning Repository. We clean the data, insert it into a normalized SQLite database, perform detailed Exploratory Data Analysis, and build predictive models (Linear Regression, Random Forest, Gradient Boosting, Stacking Ensemble, SHAP Interpretability, and GridSearchCV tuning).

In [ ]:
# Install SHAP library (required for Part 5D)
!pip install shap

## Part 1 — Dataset Selection
We choose the **Student Performance** dataset representing student grades in Mathematics. Let's download the zip, extract it, and read the raw CSV.

In [ ]:
import os
import urllib.request
import zipfile
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Create folder structure
os.makedirs('raw', exist_ok=True)
os.makedirs('gold', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Download ZIP
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00320/student.zip'
zip_path = 'raw/student.zip'
if not os.path.exists('raw/student-mat.csv'):
    print('Downloading dataset zip...')
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extract('student-mat.csv', 'raw')
    os.remove(zip_path)
    print('Extraction complete!')

# Load raw dataset (semicolon delimited)
df_raw = pd.read_csv('raw/student-mat.csv', sep=';')
print('Dataset Shape:', df_raw.shape)
df_raw.head()

In [ ]:
df_raw.describe()

## Part 2 — ETL Pipeline
We inspect the raw data, perform transformations (renaming columns, removing duplicates, imputing missing values if present), validate, and load to `gold/clean_data.csv`.

In [ ]:
# 1. Inspect
print('--- Dataset Info ---')
print(df_raw.info())
print('\n--- Missing Values ---')
print(df_raw.isnull().sum())
print('\n--- Duplicates Count ---')
print(df_raw.duplicated().sum())

# 2. Transform
df = df_raw.copy()
df.columns = [col.lower() for col in df.columns] # standard lowercase

# Drop duplicates
df = df.drop_duplicates()

# Impute missing values (just in case)
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Strip whitespace
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# 3. Validate
print('\n--- Validation Check ---')
print('Nulls remaining:', df.isnull().sum().sum())
print('Duplicates remaining:', df.duplicated().sum())
print('Cleaned Shape:', df.shape)

# 4. Load
df.to_csv('gold/clean_data.csv', index=False)
print('Saved cleaned CSV to gold/clean_data.csv')

## Part 3 — Database Schema
We design a relational schema with 4 normalized tables, run DDL statements with SQLite, and load the cleaned CSV data.

In [ ]:
import sqlite3

db_path = 'student_performance.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
cursor.execute('PRAGMA foreign_keys = ON;')

# Create tables
cursor.executescript('''
CREATE TABLE IF NOT EXISTS students (
    student_id INTEGER PRIMARY KEY AUTOINCREMENT,
    school TEXT NOT NULL,
    sex TEXT NOT NULL,
    age INTEGER NOT NULL CHECK (age BETWEEN 15 AND 22),
    address TEXT NOT NULL,
    famsize TEXT NOT NULL,
    pstatus TEXT NOT NULL
);

CREATE TABLE IF NOT EXISTS family_background (
    student_id INTEGER PRIMARY KEY,
    medu INTEGER NOT NULL CHECK (medu BETWEEN 0 AND 4),
    fedu INTEGER NOT NULL CHECK (fedu BETWEEN 0 AND 4),
    mjob TEXT NOT NULL,
    fjob TEXT NOT NULL,
    reason TEXT NOT NULL,
    guardian TEXT NOT NULL,
    FOREIGN KEY (student_id) REFERENCES students(student_id)
);

CREATE TABLE IF NOT EXISTS student_lifestyle (
    student_id INTEGER PRIMARY KEY,
    traveltime INTEGER NOT NULL CHECK (traveltime BETWEEN 1 AND 4),
    studytime INTEGER NOT NULL CHECK (studytime BETWEEN 1 AND 4),
    failures INTEGER NOT NULL CHECK (failures BETWEEN 0 AND 4),
    schoolsup TEXT NOT NULL,
    famsup TEXT NOT NULL,
    paid TEXT NOT NULL,
    activities TEXT NOT NULL,
    nursery TEXT NOT NULL,
    higher TEXT NOT NULL,
    internet TEXT NOT NULL,
    romantic TEXT NOT NULL,
    famrel INTEGER NOT NULL CHECK (famrel BETWEEN 1 AND 5),
    freetime INTEGER NOT NULL CHECK (freetime BETWEEN 1 AND 5),
    goout INTEGER NOT NULL CHECK (goout BETWEEN 1 AND 5),
    health INTEGER NOT NULL CHECK (health BETWEEN 1 AND 5),
    FOREIGN KEY (student_id) REFERENCES students(student_id)
);

CREATE TABLE IF NOT EXISTS grades_absences (
    student_id INTEGER PRIMARY KEY,
    dalc INTEGER NOT NULL CHECK (dalc BETWEEN 1 AND 5),
    walc INTEGER NOT NULL CHECK (walc BETWEEN 1 AND 5),
    absences INTEGER NOT NULL CHECK (absences >= 0),
    g1 INTEGER NOT NULL CHECK (g1 BETWEEN 0 AND 20),
    g2 INTEGER NOT NULL CHECK (g2 BETWEEN 0 AND 20),
    g3 INTEGER NOT NULL CHECK (g3 BETWEEN 0 AND 20),
    FOREIGN KEY (student_id) REFERENCES students(student_id)
);
''')
conn.commit()

# Clear old data
cursor.execute('DELETE FROM grades_absences;')
cursor.execute('DELETE FROM student_lifestyle;')
cursor.execute('DELETE FROM family_background;')
cursor.execute('DELETE FROM students;')
conn.commit()

# Insert clean data
df_clean = pd.read_csv('gold/clean_data.csv')
for idx, row in df_clean.iterrows():
    cursor.execute('INSERT INTO students (school, sex, age, address, famsize, pstatus) VALUES (?, ?, ?, ?, ?, ?);',
                   (row['school'], row['sex'], int(row['age']), row['address'], row['famsize'], row['pstatus']))
    student_id = cursor.lastrowid
    
    cursor.execute('INSERT INTO family_background (student_id, medu, fedu, mjob, fjob, reason, guardian) VALUES (?, ?, ?, ?, ?, ?, ?);',
                   (student_id, int(row['medu']), int(row['fedu']), row['mjob'], row['fjob'], row['reason'], row['guardian']))
    
    cursor.execute('INSERT INTO student_lifestyle (student_id, traveltime, studytime, failures, schoolsup, famsup, paid, activities, nursery, higher, internet, romantic, famrel, freetime, goout, health) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?);',
                   (student_id, int(row['traveltime']), int(row['studytime']), int(row['failures']), row['schoolsup'], row['famsup'], row['paid'], row['activities'], row['nursery'], row['higher'], row['internet'], row['romantic'], int(row['famrel']), int(row['freetime']), int(row['goout']), int(row['health'])))
    
    cursor.execute('INSERT INTO grades_absences (student_id, dalc, walc, absences, g1, g2, g3) VALUES (?, ?, ?, ?, ?, ?, ?);',
                   (student_id, int(row['dalc']), int(row['walc']), int(row['absences']), int(row['g1']), int(row['g2']), int(row['g3'])))

conn.commit()

print('--- Verification Summary ---')
for table in ['students', 'family_background', 'student_lifestyle', 'grades_absences']:
    cursor.execute(f'SELECT COUNT(*) FROM {table};')
    print(f"Table '{table}' row count: {cursor.fetchone()[0]}")
conn.close()

## Part 4 — Exploratory Data Analysis
### 4A: Descriptive Statistics

In [ ]:
df = pd.read_csv('gold/clean_data.csv')
print('Descriptive stats of numerical columns:')
display(df.describe())
print('\nWeekly study time distribution:')
display(df['studytime'].value_counts())
print('\nMissing values count:')
display(df.isnull().sum())

### 4B: Univariate Analysis (Numerical Histograms & Boxplots, Categorical Counts)

In [ ]:
# Numerical Histograms
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
sns.histplot(df['age'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of Age')
sns.histplot(df['absences'], kde=True, ax=axes[0, 1], color='salmon')
axes[0, 1].set_title('Distribution of Absences')
sns.histplot(df['g1'], kde=True, ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Distribution of G1 Grade')
sns.histplot(df['g3'], kde=True, ax=axes[1, 1], color='violet')
axes[1, 1].set_title('Distribution of G3 (Target Grade)')
plt.tight_layout()
plt.savefig('outputs/univariate_numerical.png', dpi=150)
plt.show()

# Boxplots to see outliers
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
sns.boxplot(y=df['age'], ax=axes[0], color='skyblue')
axes[0].set_title('Outliers in Age')
sns.boxplot(y=df['absences'], ax=axes[1], color='salmon')
axes[1].set_title('Outliers in Absences')
sns.boxplot(y=df['g3'], ax=axes[2], color='violet')
axes[2].set_title('Outliers in G3 (Target)')
plt.tight_layout()
plt.savefig('outputs/univariate_numerical_boxplots.png', dpi=150)
plt.show()

# Categorical bar charts
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.countplot(x='mjob', data=df, ax=axes[0], palette='viridis')
axes[0].set_title("Mother's Job Counts")
axes[0].tick_params(axis='x', rotation=45)
sns.countplot(x='fjob', data=df, ax=axes[1], palette='plasma')
axes[1].set_title("Father's Job Counts")
axes[1].tick_params(axis='x', rotation=45)
sns.countplot(x='studytime', data=df, ax=axes[2], palette='magma')
axes[2].set_title('Weekly Study Time')
plt.tight_layout()
plt.savefig('outputs/univariate_categorical.png', dpi=150)
plt.show()

### 4C: Bivariate & Multivariate Analysis

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 10))
numeric_cols = df.select_dtypes(include=[np.number]).columns
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numerical Features')
plt.savefig('outputs/correlation_heatmap.png', dpi=150)
plt.show()

# Scatter plots vs Target G3
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.scatterplot(data=df, x='g1', y='g3', ax=axes[0], color='green', alpha=0.6)
axes[0].set_title('G1 vs G3 Grade')
sns.scatterplot(data=df, x='g2', y='g3', ax=axes[1], color='blue', alpha=0.6)
axes[1].set_title('G2 vs G3 Grade')
sns.scatterplot(data=df, x='absences', y='g3', ax=axes[2], color='red', alpha=0.6)
axes[2].set_title('Absences vs G3 Grade')
plt.tight_layout()
plt.savefig('outputs/bivariate_scatter.png', dpi=150)
plt.show()

# Box plots vs Target G3
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
sns.boxplot(data=df, x='studytime', y='g3', ax=axes[0], palette='Blues')
axes[0].set_title('Study Time vs G3 Grade')
sns.boxplot(data=df, x='romantic', y='g3', ax=axes[1], palette='Purples')
axes[1].set_title('Romantic Status vs G3 Grade')
sns.boxplot(data=df, x='schoolsup', y='g3', ax=axes[2], palette='Oranges')
axes[2].set_title('School Support vs G3 Grade')
plt.tight_layout()
plt.savefig('outputs/bivariate_boxplot.png', dpi=150)
plt.show()

# 3x2 Combined Subplot Dashboard
fig, axes = plt.subplots(3, 2, figsize=(14, 16))
sns.heatmap(df[numeric_cols].corr(), cmap='coolwarm', ax=axes[0, 0])
axes[0, 0].set_title('Correlation Heatmap')
sns.scatterplot(data=df, x='g2', y='g3', ax=axes[0, 1], color='indigo', alpha=0.7)
axes[0, 1].set_title('G2 Grade vs G3 Grade (Strong Predictor)')
sns.boxplot(data=df, x='studytime', y='g3', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Study Time vs G3 Grade')
sns.boxplot(data=df, x='failures', y='g3', ax=axes[1, 1], palette='Reds')
axes[1, 1].set_title('Past Class Failures vs G3 Grade')
sns.boxplot(data=df, x='schoolsup', y='g3', ax=axes[2, 0], palette='Set1')
axes[2, 0].set_title('School Support vs G3 Grade')
sns.boxplot(data=df, x='romantic', y='g3', ax=axes[2, 1], palette='Pastel1')
axes[2, 1].set_title('Romantic relationship vs G3 Grade')
plt.tight_layout()
plt.savefig('outputs/eda_dashboard.png', dpi=150)
plt.show()

### 4D: Hypothesis Testing

In [ ]:
print('HYPOTHESIS 1: Students with higher study time will score better in G3.')
high_study = df[df['studytime'] >= 3]['g3']
low_study = df[df['studytime'] <= 2]['g3']
t_stat, p_val = stats.ttest_ind(high_study, low_study)
print(f'T-statistic: {t_stat:.3f}, p-value: {p_val:.6f}')
print('Result:', 'CONFIRMED' if p_val < 0.05 else 'REJECTED')

print('\nHYPOTHESIS 2: Students in a romantic relationship have lower G3 scores.')
rom_yes = df[df['romantic'] == 'yes']['g3']
rom_no = df[df['romantic'] == 'no']['g3']
t_stat, p_val = stats.ttest_ind(rom_yes, rom_no)
print(f'T-statistic: {t_stat:.3f}, p-value: {p_val:.6f}')
print('Result:', 'CONFIRMED' if p_val < 0.05 else 'REJECTED')

print('\nHYPOTHESIS 3: Students receiving extra school educational support have lower G3 scores.')
sup_yes = df[df['schoolsup'] == 'yes']['g3']
sup_no = df[df['schoolsup'] == 'no']['g3']
t_stat, p_val = stats.ttest_ind(sup_yes, sup_no)
print(f'T-statistic: {t_stat:.3f}, p-value: {p_val:.6f}')
print('Result:', 'CONFIRMED' if p_val < 0.05 else 'REJECTED')

print('\nHYPOTHESIS 4: Students with higher weekend alcohol consumption (Walc) have higher absences.')
corr, p_val = stats.pearsonr(df['walc'], df['absences'])
print(f'Pearson correlation: {corr:.3f}, p-value: {p_val:.6f}')
print('Result:', 'CONFIRMED' if p_val < 0.05 else 'REJECTED')

## Part 5 — Predictive Model Development
### 5A: Model Architecture Selection

In [ ]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error

# Prepare data (One-hot encoding)
df_ml = pd.get_dummies(df, drop_first=True)
X = df_ml.drop('g3', axis=1)
y = df_ml['g3']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models to compare
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

# Evaluate with 5-Fold Cross-Validation
print('--- 5-Fold CV Performance ---')
for name, model in models.items():
    cv = cross_validate(model, X_train, y_train, cv=5, scoring=('r2', 'neg_mean_absolute_error'))
    print(f"{name}:")
    print(f"  Mean R2 Score: {np.mean(cv['test_r2']):.4f}")
    print(f"  Mean MAE: {-np.mean(cv['test_neg_mean_absolute_error']):.4f}")

### 5B: Feature Importance Assessment

In [ ]:
# Train best model (Random Forest)
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)

# Calculate importances
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
feature_names = X.columns

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices[:10]], y=feature_names[indices[:10]], palette='viridis')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Relative Importance')
plt.savefig('outputs/feature_importance.png', dpi=150)
plt.show()

# Identify features with importance < 0.01
low_importance = [feature_names[i] for i in range(len(importances)) if importances[i] < 0.01]
print(f"Total features with importance < 0.01: {len(low_importance)}")
print("Sample of removable features:", low_importance[:10])

### 5C: Ensemble Model Creation

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

# Define base models
estimators = [
    ('lr', LinearRegression()),
    ('rf', RandomForestRegressor(random_state=42)),
    ('gb', GradientBoostingRegressor(random_state=42))
]

# Create Stacking Ensemble with Ridge as Meta-Model
stack_model = StackingRegressor(estimators=estimators, final_estimator=Ridge())
stack_model.fit(X_train, y_train)

# Compare performance on test set
results = []
for name, model in [('Linear Regression', LinearRegression()), 
                    ('Random Forest', RandomForestRegressor(random_state=42)), 
                    ('Gradient Boosting', GradientBoostingRegressor(random_state=42)),
                    ('Stacking Ensemble', stack_model)]:
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Test R2': r2_score(y_test, pred),
        'Test MAE': mean_absolute_error(y_test, pred)
    })

display(pd.DataFrame(results))

### 5D: Model Interpretability (SHAP)

In [ ]:
import shap

# Compute SHAP values
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_train)

# Beeswarm plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_train, show=False)
plt.title('SHAP Summary Beeswarm Plot', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/shap_summary.png', dpi=150)
plt.show()

# Force plot for Row 0 student
shap.initjs()
plt.figure(figsize=(12, 4))
shap.force_plot(explainer.expected_value, shap_values[0, :], X_train.iloc[0, :], matplotlib=True, show=False)
plt.title('SHAP Force Plot for Row 0 Student', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/shap_force.png', dpi=150)
plt.show()

### 5E: Improved Generalisation

In [ ]:
from sklearn.model_selection import GridSearchCV

# Check overfitting
train_r2 = rf_model.score(X_train, y_train)
test_r2 = rf_model.score(X_test, y_test)
print(f"Default RF Train R2: {train_r2:.4f}, Test R2: {test_r2:.4f}")
print(f"Overfit Gap: {train_r2 - test_r2:.4f}")

# GridSearchCV Tuning
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [4, 6, 8],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=5, scoring='r2')
grid_search.fit(X_train, y_train)

best_rf_model = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)

# Evaluation after tuning
train_r2_after = best_rf_model.score(X_train, y_train)
test_r2_after = best_rf_model.score(X_test, y_test)

tuning_comparison = pd.DataFrame({
    'Metric': ['Train R2', 'Test R2', 'Overfit Gap'],
    'Before Tuning': [train_r2, test_r2, train_r2 - test_r2],
    'After Tuning': [train_r2_after, test_r2_after, train_r2_after - test_r2_after]
})
display(tuning_comparison)

## Part 6 — Create Prediction from the Final model
We create a new student representation, update their study time to the maximum value (4), and predict their final grade G3 using our tuned Random Forest model.

In [ ]:
# Take a sample student
new_student = X_train.iloc[[0]].copy()
print('Original student study time:', new_student['studytime'].values[0])

# Change study time to 4 (maximum)
new_student.iloc[0, X_train.columns.get_loc('studytime')] = 4
print('Modified student study time:', new_student['studytime'].values[0])

# Predict
predicted_grade = best_rf_model.predict(new_student)[0]
print(f'\nModel Predicted G3 Grade: {predicted_grade:.2f} (scale 0-20)')